---
## 📦 Install Required Libraries


In [ ]:
# Install necessary packages
!pip install reportlab pillow --quiet
print("✅ All packages installed successfully!")

✅ All packages installed successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## 🔧 Import Libraries

In [ ]:
# Standard libraries
import os
import io
import base64
from datetime import datetime
from PIL import Image
import re

# Data science libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2

# TensorFlow
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# UI components
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Javascript

# PDF generation
from reportlab.lib.pagesizes import letter, A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image as RLImage, PageBreak
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY
from reportlab.pdfgen import canvas

# Google Colab file handling
from google.colab import files

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


---
## 🧠 Load Your Trained Model

In [ ]:
# Load your trained ResNet50 model
# MODIFY THIS PATH to match your model location
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/CKD_MODELS/resnet50_final.keras",
    compile=False
)

CLASS_NAMES = ["Cyst", "Normal", "Stone", "Tumor"]

print("✅ Model loaded successfully!")
print(f"Model expects input shape: {model.input_shape}")

✅ Model loaded successfully!
Model expects input shape: (None, 224, 224, 3)


---
## 🔍 Grad-CAM Visualization Functions

In [ ]:
import tensorflow as tf
import numpy as np
import cv2


def make_gradcam_heatmap(img_array, model):
    """
    Grad-CAM using backbone submodel — exactly as used in working main notebook.
    backbone.input + tf.reduce_mean(features) is the pattern that works.
    """
    backbone = model.get_layer("resnet50")
    last_conv_layer = backbone.get_layer("conv5_block3_out")

    grad_model = tf.keras.models.Model(
        inputs=backbone.input,
        outputs=[last_conv_layer.output, backbone.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, features = grad_model(img_array)
        loss = tf.reduce_mean(features)

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap) + 1e-8

    return heatmap


def overlay_heatmap(img_array_float, heatmap, alpha=0.35):
    """
    Overlay Grad-CAM heatmap on image.
    Matches the exact overlay logic from the working main notebook:
      cv2.addWeighted(orig, 0.65, heatmap_color, 0.35, 0)

    img_array_float : float32 RGB in [0,1], shape (224,224,3)
    heatmap         : float32 from make_gradcam_heatmap, shape (7,7)
    Returns         : uint8 RGB overlay, shape (224,224,3)
    """
    # Convert original to uint8 BGR for cv2
    orig_bgr = cv2.cvtColor(np.uint8(img_array_float * 255), cv2.COLOR_RGB2BGR)

    # Resize and colorize heatmap
    heatmap_resized = cv2.resize(np.float32(heatmap), (224, 224))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

    # Blend — same weights as working main notebook
    overlay_bgr = cv2.addWeighted(orig_bgr, 0.65, heatmap_color, 0.35, 0)

    # Return as RGB for matplotlib
    return cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)


print("✅ Grad-CAM functions loaded!")


✅ Grad-CAM functions loaded!


---
## 📚 Knowledge Base (Same as Before)

In [ ]:
KIDNEY_KNOWLEDGE_BASE = {
    "Normal": {
        "summary": "Your kidneys appear healthy with no visible abnormalities.",
        "description": "Normal kidneys show typical anatomical structure without cysts, stones, or tumors. The cortex and medulla are well-defined, and there's no evidence of obstruction or masses.",
        "next_steps": [
            "Continue regular health checkups",
            "Maintain adequate hydration (8-10 glasses of water daily)",
            "Follow a balanced diet low in sodium",
            "Monitor blood pressure regularly",
            "Avoid excessive use of NSAIDs (pain medications)"
        ],
        "follow_up": "Annual routine checkup recommended",
        "lifestyle_tips": [
            "Stay hydrated throughout the day",
            "Limit salt intake to less than 2,300mg per day",
            "Exercise regularly (150 minutes per week)",
            "Avoid smoking and excessive alcohol",
            "Maintain healthy body weight"
        ],
        "urgency_level": "Routine",
        "urgency_color": "#4caf50"
    },
    "Cyst": {
        "summary": "A fluid-filled sac detected in the kidney tissue.",
        "description": "Kidney cysts are round pouches of fluid that form on or in the kidneys. Simple cysts are common and usually harmless, especially in older adults. They typically don't cause symptoms unless they grow large.",
        "next_steps": [
            "Consult a nephrologist or urologist for evaluation",
            "May need follow-up imaging (ultrasound or CT) in 6-12 months",
            "Monitor for symptoms: pain, fever, or blood in urine",
            "Evaluate kidney function with blood tests (creatinine, GFR)",
            "Large or symptomatic cysts may require drainage or surgery"
        ],
        "symptoms_to_watch": [
            "Persistent back or side pain",
            "Fever (could indicate infection)",
            "Blood in urine",
            "Frequent urination",
            "High blood pressure"
        ],
        "follow_up": "Follow-up imaging in 6-12 months, sooner if symptomatic",
        "lifestyle_tips": [
            "Stay well-hydrated",
            "Monitor blood pressure regularly",
            "Avoid kidney-stressing medications unless prescribed",
            "Report new symptoms immediately"
        ],
        "urgency_level": "Follow-up Needed",
        "urgency_color": "#ff9800"
    },
    "Stone": {
        "summary": "A hard mineral deposit detected in the kidney.",
        "description": "Kidney stones are hard deposits made of minerals and salts that form inside your kidneys. They can cause severe pain when passing through the urinary tract. Size, location, and composition determine treatment.",
        "next_steps": [
            "Immediate urologist consultation",
            "Pain management (prescribed medications)",
            "Increase water intake to 2-3 liters daily",
            "Stone analysis to determine composition",
            "Consider lithotripsy or ureteroscopy for large stones",
            "24-hour urine collection to identify risk factors"
        ],
        "symptoms_to_watch": [
            "Severe, sharp pain in back/side/lower abdomen",
            "Pain during urination",
            "Pink, red, or brown urine",
            "Nausea and vomiting",
            "Fever and chills (seek emergency care)"
        ],
        "prevention_tips": [
            "Drink 2.5-3 liters of water daily",
            "Limit sodium intake",
            "Reduce animal protein consumption",
            "Limit oxalate-rich foods if calcium oxalate stones"
        ],
        "follow_up": "Immediate urologist appointment required",
        "urgency_level": "Prompt Action Required",
        "urgency_color": "#ff5722"
    },
    "Tumor": {
        "summary": "An abnormal growth detected in the kidney tissue.",
        "description": "A kidney tumor is an abnormal growth that can be benign or malignant. The most common kidney cancer is renal cell carcinoma (RCC). Early detection significantly improves treatment outcomes.",
        "next_steps": [
            "URGENT: Immediate oncologist and urologist consultation",
            "Additional imaging: MRI or contrast-enhanced CT for staging",
            "Biopsy may be recommended to determine tumor type",
            "Blood tests: CBC, kidney function, tumor markers",
            "Multidisciplinary team evaluation for treatment planning"
        ],
        "symptoms_to_watch": [
            "Blood in urine (hematuria)",
            "Persistent back or side pain",
            "Palpable mass in abdomen",
            "Unexplained weight loss",
            "Fever and fatigue"
        ],
        "follow_up": "URGENT - Schedule oncology consultation within 1-2 weeks",
        "urgency_level": "Urgent - Immediate Action",
        "urgency_color": "#f44336"
    }
}

print("✅ Knowledge base loaded!")

✅ Knowledge base loaded!


---
## 📄 PDF Report Generator

In [ ]:
class KidneyDiagnosticReport:
    """
    Optimized PDF generator - EXACTLY 2 pages
    Page 1: Header + Diagnosis + Images
    Page 2: Clinical Recommendations
    """

    def __init__(self, patient_name="Patient"):
        self.patient_name = patient_name
        self.report_date = datetime.now().strftime("%B %d, %Y at %I:%M %p")
        self.styles = getSampleStyleSheet()
        self._setup_styles()

    def _setup_styles(self):
        self.styles.add(ParagraphStyle(
            'CustomTitle', parent=self.styles['Heading1'], fontSize=22,
            textColor=colors.HexColor('#1976d2'), spaceAfter=20, alignment=TA_CENTER, fontName='Helvetica-Bold'))

        self.styles.add(ParagraphStyle(
            'SectionHeader', parent=self.styles['Heading2'], fontSize=13,
            textColor=colors.HexColor('#424242'), spaceAfter=10, spaceBefore=15, fontName='Helvetica-Bold'))

        self.styles.add(ParagraphStyle(
            'CustomBody', parent=self.styles['Normal'], fontSize=10, leading=14, alignment=TA_JUSTIFY, spaceAfter=8))

    def generate_report(self, predicted_class, confidence, original_img_path, gradcam_img_path,
                       filename="kidney_diagnostic_report.pdf"):
        """
        Generate optimized 2-page PDF report
        """
        doc = SimpleDocTemplate(filename, pagesize=letter, rightMargin=60, leftMargin=60,
                               topMargin=60, bottomMargin=40)
        story = []
        condition_info = KIDNEY_KNOWLEDGE_BASE.get(predicted_class, {})
        urgency_color = colors.HexColor(condition_info.get('urgency_color', '#757575'))

        # ===== PAGE 1 =====

        # HEADER
        story.append(Paragraph("KIDNEY LESION DIAGNOSTIC REPORT", self.styles['CustomTitle']))
        story.append(Spacer(1, 8))

        # Patient Info Table (compact)
        info_data = [
            ['Patient Name:', self.patient_name, 'Report Date:', self.report_date],
            ['Analysis Method:', 'AI-Powered CNN (ResNet50)', 'Report ID:',
             f'KDN-{datetime.now().strftime("%Y%m%d%H%M%S")}']
        ]
        info_table = Table(info_data, colWidths=[1.2*inch, 2*inch, 1*inch, 2*inch])
        info_table.setStyle(TableStyle([
            ('FONT', (0, 0), (0, -1), 'Helvetica-Bold', 9),
            ('FONT', (2, 0), (2, -1), 'Helvetica-Bold', 9),
            ('FONT', (1, 0), (1, -1), 'Helvetica', 9),
            ('FONT', (3, 0), (3, -1), 'Helvetica', 9),
            ('TEXTCOLOR', (0, 0), (-1, -1), colors.HexColor('#424242')),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
        ]))
        story.append(info_table)
        story.append(Spacer(1, 12))

        # DIAGNOSIS
        story.append(Paragraph("DIAGNOSTIC FINDINGS", self.styles['SectionHeader']))

        diagnosis_data = [
            ['Detected Condition:', predicted_class],
            ['Model Confidence:', f'{confidence:.1f}%'],
            ['Urgency Level:', condition_info.get('urgency_level', 'N/A')]
        ]
        diagnosis_table = Table(diagnosis_data, colWidths=[2*inch, 4.2*inch])
        diagnosis_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor('#f5f5f5')),
            ('FONT', (0, 0), (0, -1), 'Helvetica-Bold', 10),
            ('FONT', (1, 0), (1, -1), 'Helvetica-Bold', 10),
            ('TEXTCOLOR', (1, 0), (1, 0), urgency_color),
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('PADDING', (0, 0), (-1, -1), 10),
            ('BOX', (0, 0), (-1, -1), 2, urgency_color),
        ]))
        story.append(diagnosis_table)
        story.append(Spacer(1, 10))

        # Summary & Description (compact)
        story.append(Paragraph("<b>Summary:</b>", self.styles['CustomBody']))
        story.append(Paragraph(condition_info.get('summary', ''), self.styles['CustomBody']))
        story.append(Spacer(1, 6))
        story.append(Paragraph("<b>Description:</b>", self.styles['CustomBody']))
        story.append(Paragraph(condition_info.get('description', ''), self.styles['CustomBody']))
        story.append(Spacer(1, 12))

        # IMAGES
        story.append(Paragraph("IMAGING ANALYSIS", self.styles['SectionHeader']))

        if os.path.exists(original_img_path) and os.path.exists(gradcam_img_path):
            img_width = 2.8 * inch
            img_height = 2.8 * inch

            img_data = [
                [RLImage(original_img_path, width=img_width, height=img_height),
                 RLImage(gradcam_img_path, width=img_width, height=img_height)],
                [Paragraph('<b>Original CT Scan</b>', self.styles['Normal']),
                 Paragraph('<b>Grad-CAM Visualization</b>', self.styles['Normal'])]
            ]
            img_table = Table(img_data, colWidths=[3.2*inch, 3.2*inch])
            img_table.setStyle(TableStyle([
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
            ]))
            story.append(img_table)

        # PAGE BREAK - Start Page 2
        story.append(PageBreak())

        # ===== PAGE 2 =====

        story.append(Paragraph("CLINICAL RECOMMENDATIONS", self.styles['SectionHeader']))

        # Next Steps
        story.append(Paragraph("<b>Recommended Next Steps:</b>", self.styles['CustomBody']))
        for i, step in enumerate(condition_info.get('next_steps', []), 1):
            story.append(Paragraph(f"{i}. {step}", self.styles['CustomBody']))
        story.append(Spacer(1, 10))

        # Follow-up
        story.append(Paragraph("<b>Follow-up Schedule:</b>", self.styles['CustomBody']))
        story.append(Paragraph(condition_info.get('follow_up', ''), self.styles['CustomBody']))
        story.append(Spacer(1, 10))

        # Symptoms
        if 'symptoms_to_watch' in condition_info:
            story.append(Paragraph("<b>Symptoms to Monitor:</b>", self.styles['CustomBody']))
            for symptom in condition_info['symptoms_to_watch']:
                story.append(Paragraph(f"• {symptom}", self.styles['CustomBody']))
            story.append(Spacer(1, 10))

        # Lifestyle
        if 'lifestyle_tips' in condition_info or 'prevention_tips' in condition_info:
            tips = condition_info.get('lifestyle_tips', condition_info.get('prevention_tips', []))
            story.append(Paragraph("<b>Lifestyle Recommendations:</b>", self.styles['CustomBody']))
            for tip in tips[:5]:  # Limit to 5 to save space
                story.append(Paragraph(f"• {tip}", self.styles['CustomBody']))
            story.append(Spacer(1, 15))

        # DISCLAIMER (compact)
        disclaimer = """<b>IMPORTANT DISCLAIMER:</b> This report is generated by an AI-powered diagnostic support
        system and is intended for informational purposes only. It does not constitute a medical diagnosis or
        replace professional medical advice. All findings should be reviewed and confirmed by qualified healthcare
        professionals. The AI model's predictions are based on CT imaging analysis and should be interpreted in
        conjunction with clinical examination, patient history, and additional diagnostic tests. Please consult
        with your physician or specialist for proper diagnosis and treatment recommendations."""

        disclaimer_style = ParagraphStyle('Disclaimer', parent=self.styles['Normal'], fontSize=8,
            textColor=colors.HexColor('#757575'), alignment=TA_JUSTIFY, borderColor=colors.HexColor('#e0e0e0'),
            borderWidth=1, borderPadding=8, backColor=colors.HexColor('#fafafa'))
        story.append(Paragraph(disclaimer, disclaimer_style))

        # FOOTER
        story.append(Spacer(1, 10))
        footer = f"""<i>Report generated by AI-Powered Kidney Lesion Classification System<br/>
        Developed as part of medical imaging research project | Generated on: {self.report_date}</i>"""
        footer_style = ParagraphStyle('Footer', parent=self.styles['Normal'], fontSize=7,
            textColor=colors.HexColor('#9e9e9e'), alignment=TA_CENTER)
        story.append(Paragraph(footer, footer_style))

        doc.build(story)
        print(f"✅ 2-page PDF report generated: {filename}")
        return filename

print("✅ Optimized 2-page PDF generator loaded!")

✅ Optimized 2-page PDF generator loaded!


---
## 🤖 Improved Chatbot (Same as Before)

In [ ]:
class ImprovedKidneyChatbot:
    """
    FINAL FIXED VERSION - Clean formatting in responses
    """

    def __init__(self, predicted_class, confidence_score):
        self.predicted_class = predicted_class
        self.confidence_score = confidence_score
        self.conversation_history = []
        self.last_topic = None
        self.chat_output = widgets.Output()
        self.user_input = widgets.Text(
            placeholder='Ask me anything about your kidney condition...',
            layout=widgets.Layout(width='80%')
        )
        self.send_button = widgets.Button(
            description='Send',
            button_style='primary',
            icon='paper-plane',
            layout=widgets.Layout(width='15%')
        )

    def _add_message(self, sender, message):
        """Display message in chat with proper HTML formatting"""
        timestamp = datetime.now().strftime("%H:%M")

        if sender == "AI":
            style = "background: #e3f2fd; border-left: 4px solid #2196f3;"
            icon = "🤖"
        else:
            style = "background: #f5f5f5; border-left: 4px solid #757575;"
            icon = "👤"

        # Convert markdown-style formatting to HTML
        message_html = message.replace('**', '<strong>').replace('**', '</strong>')
        message_html = message_html.replace('\n', '<br>')

        with self.chat_output:
            display(HTML(f"""
                <div style="{style} margin: 10px 0; padding: 12px; border-radius: 8px;">
                    <div style="font-weight: bold; color: #424242; margin-bottom: 5px;">
                        {icon} {sender} <span style="font-size: 0.8em; color: #9e9e9e; font-weight: normal;">{timestamp}</span>
                    </div>
                    <div style="color: #212121; line-height: 1.6;">
                        {message_html}
                    </div>
                </div>
            """))

    def _classify_question(self, question):
        """Classify question type with priority ordering"""
        q_lower = question.lower()

        if re.search(r'(treatment|cure|fix|heal|surgery|operation|procedure|therapy|medication)', q_lower):
            return 'treatment'
        if re.search(r'(emergency|urgent|911|er\\b)', q_lower):
            return 'emergency'
        if re.search(r'(symptom|sign|feel|notice|watch for)', q_lower):
            return 'symptoms'
        if re.search(r'(when.*doctor|see.*doctor|appointment)', q_lower):
            return 'when_doctor'
        if re.search(r'(what.*next|do now|should i do|step)', q_lower):
            return 'next_steps'
        if re.search(r'(type|kind|different)', q_lower):
            return 'types'
        if re.search(r'(worry|concern|serious|scared|dangerous)', q_lower):
            return 'worry'
        if re.search(r'(confident|sure|accurate|reliable)', q_lower):
            return 'confidence'
        if re.search(r'(prevent|avoid|lifestyle|diet|food|eat)', q_lower):
            return 'prevention'
        if re.search(r'(cost|price|expensive|afford)', q_lower):
            return 'cost'
        if re.search(r'(what is|what does|explain|mean)', q_lower):
            return 'explanation'

        return 'general'

    def _generate_response(self, user_message):
        """Generate clean, properly formatted responses"""
        condition = KIDNEY_KNOWLEDGE_BASE.get(self.predicted_class, {})
        question_type = self._classify_question(user_message)

        if question_type == 'treatment':
            if 'treatment_options' in condition:
                options = condition['treatment_options']
                response = f"<strong>Treatment Options for {self.predicted_class}:</strong><br><br>"
                response += "<br>".join(f"• {opt}" for opt in options)
                response += "<br><br>⚕️ Your doctor will recommend the best treatment based on your specific case."
            else:
                if self.predicted_class == "Normal":
                    response = "Since your kidneys appear healthy, no treatment is needed. Focus on preventive care."
                else:
                    response = f"Treatment for {self.predicted_class} will be determined by your healthcare team based on your specific situation."
            return response

        elif question_type == 'prevention':
            tips = condition.get('prevention_tips', condition.get('lifestyle_tips', [
                "Stay well-hydrated (8-10 glasses daily)",
                "Eat a balanced diet",
                "Exercise regularly",
                "Limit salt intake",
                "Avoid smoking"
            ]))
            response = f"<strong>Lifestyle Recommendations for {self.predicted_class}:</strong><br><br>"
            response += "<br>".join(f"• {tip}" for tip in tips)
            response += "<br><br>🥗 These habits support kidney health and may help prevent future issues."
            return response

        elif question_type == 'symptoms':
            symptoms = condition.get('symptoms_to_watch', [])
            if symptoms:
                response = f"<strong>Symptoms to Watch For ({self.predicted_class}):</strong><br><br>"
                response += "<br>".join(f"• {s}" for s in symptoms)
                response += "<br><br>⚠️ Contact your doctor if you notice any of these symptoms."
            else:
                response = "Watch for any unusual pain, changes in urination, or concerning symptoms."
            return response

        elif question_type == 'next_steps':
            steps = condition.get('next_steps', [])
            response = f"<strong>Recommended Next Steps for {self.predicted_class}:</strong><br><br>"
            response += "<br>".join(f"{i+1}. {step}" for i, step in enumerate(steps))
            response += "<br><br>⚠️ These are general guidelines. Your doctor will provide personalized advice."
            return response

        elif question_type == 'when_doctor':
            follow_up = condition.get('follow_up', 'Consult your healthcare provider')
            response = f"<strong>Recommended Timeline:</strong> {follow_up}<br><br>"
            response += "<strong>📅 When you visit, bring:</strong><br>"
            response += "• Your CT scan images/report<br>"
            response += "• List of current medications<br>"
            response += "• Any symptoms you're experiencing<br><br>"
            response += "Don't delay scheduling this appointment."
            return response

        elif question_type == 'worry':
            if self.predicted_class == "Normal":
                response = "✅ Good news! Your kidneys appear healthy. Continue with preventive care - no cause for concern."
            elif self.predicted_class == "Cyst":
                response = "Most kidney cysts are benign. While they need evaluation, try not to worry excessively. Many people live normally with kidney cysts that just need monitoring."
            elif self.predicted_class == "Stone":
                response = "Kidney stones are painful but treatable. Many pass naturally, and larger ones can be treated with procedures. With proper care, most people recover fully."
            elif self.predicted_class == "Tumor":
                response = "I understand this is concerning. Important facts: not all tumors are cancerous, and early detection means better treatment options. Schedule an urgent consultation with specialists."
            return response

        elif question_type == 'types':
            types = condition.get('types', [])
            if types:
                response = f"<strong>Types of {self.predicted_class}:</strong><br><br>"
                response += "<br>".join(f"• {t}" for t in types)
                response += "<br><br>Your healthcare provider can determine which type through examination."
            else:
                response = condition.get('description', '')
            return response

        elif question_type == 'emergency':
            emergency_signs = condition.get('emergency_signs', [
                "Severe, uncontrollable pain",
                "High fever with chills",
                "Inability to urinate"
            ])
            response = "🚨 <strong>SEEK EMERGENCY CARE IMMEDIATELY if you have:</strong><br><br>"
            response += "<br>".join(f"🔴 {sign}" for sign in emergency_signs)
            response += "<br><br>Don't wait - go to the ER or call emergency services!"
            return response

        elif question_type == 'confidence':
            level = "high" if self.confidence_score >= 85 else "moderate" if self.confidence_score >= 70 else "preliminary"
            response = f"<strong>AI Model Confidence:</strong> {self.confidence_score:.1f}% ({level})<br><br>"
            response += "AI analysis is a supportive tool, not a diagnosis. A healthcare professional must review images, consider your history, and provide expert opinion."
            return response

        elif question_type == 'explanation':
            response = f"<strong>Understanding {self.predicted_class}:</strong><br><br>"
            response += f"{condition.get('description', '')}<br><br>"
            response += f"<strong>Summary:</strong> {condition.get('summary', '')}"
            return response

        elif question_type == 'cost':
            response = "💰 <strong>Regarding costs:</strong><br><br>"
            response += "• Discuss cost-effective options with your doctor<br>"
            response += "• Check insurance coverage<br>"
            response += "• Ask about payment plans<br>"
            response += "• Look into patient assistance programs<br><br>"
            response += "Don't let cost prevent necessary care - options are often available."
            return response

        else:
            response = f"I can help with questions about <strong>{self.predicted_class}</strong>. Ask me about:<br><br>"
            response += "• What this condition means<br>"
            response += "• Symptoms to watch for<br>"
            response += "• Treatment options<br>"
            response += "• Prevention and lifestyle<br>"
            response += "• Next steps<br><br>"
            response += "What would you like to know?"
            return response

    def _send_message(self, button):
        """Handle message sending"""
        user_message = self.user_input.value.strip()
        if not user_message:
            return

        self._add_message("You", user_message)
        self.user_input.value = ""

        ai_response = self._generate_response(user_message)
        self._add_message("AI Assistant", ai_response)

        self.conversation_history.append({'user': user_message, 'assistant': ai_response})

    def display(self):
        """Display chatbot interface"""
        confidence_emoji = "✅" if self.confidence_score >= 85 else "⚠️" if self.confidence_score >= 70 else "❓"

        greeting = f"Hello! I'm your kidney health assistant. {confidence_emoji}<br><br>"
        greeting += f"<strong>Finding:</strong> {self.predicted_class} (Confidence: {self.confidence_score:.1f}%)<br><br>"
        greeting += "I can answer questions about:<br>"
        greeting += "• What this condition means<br>"
        greeting += "• Symptoms to watch<br>"
        greeting += "• Treatment options<br>"
        greeting += "• Prevention tips<br>"
        greeting += "• Next steps<br><br>"
        greeting += "<em>Note: This is educational information only. Consult healthcare professionals for medical decisions.</em><br><br>"
        greeting += "What would you like to know?"

        self._add_message("AI Assistant", greeting)

        self.send_button.on_click(self._send_message)
        self.user_input.on_submit(lambda x: self._send_message(None))

        input_box = widgets.HBox([self.user_input, self.send_button], layout=widgets.Layout(margin='10px 0'))

        header = widgets.HTML("""
            <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        padding: 20px; border-radius: 10px 10px 0 0; color: white; text-align: center;">
                <h2 style="margin: 0;">🩺 Kidney Health AI Assistant</h2>
                <p style="margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;">Clean Responses - Fixed Formatting</p>
            </div>
        """)

        chat_container = widgets.VBox([self.chat_output],
                                      layout=widgets.Layout(border='1px solid #ddd', padding='15px',
                                                           height='500px', overflow_y='auto', background='white'))

        full_interface = widgets.VBox([header, chat_container, input_box],
                                      layout=widgets.Layout(border='1px solid #ddd', border_radius='10px',
                                                           margin='20px 0', box_shadow='0 4px 6px rgba(0,0,0,0.1)'))
        display(full_interface)

print("✅ FIXED Chatbot with clean formatting loaded!")

✅ FIXED Chatbot with clean formatting loaded!


---
## 🎨 Complete UI System

In [ ]:
class KidneyDiagnosisUI:
    """
    Complete professional UI for kidney lesion classification
    """

    def __init__(self, model):
        self.model = model
        self.uploaded_file = None
        self.prediction_result = None
        self.original_img_path = None
        self.gradcam_img_path = None

        self.output_area = widgets.Output()
        self.upload_btn = widgets.Button(
            description='📁 Upload CT Scan',
            button_style='info',
            layout=widgets.Layout(width='200px', height='45px')
        )
        self.analyze_btn = widgets.Button(
            description='🔬 Analyze Image',
            button_style='success',
            layout=widgets.Layout(width='200px', height='45px'),
            disabled=True
        )
        self.download_btn = widgets.Button(
            description='📄 Download PDF Report',
            button_style='warning',
            layout=widgets.Layout(width='200px', height='45px'),
            disabled=True
        )
        self.patient_name_input = widgets.Text(
            placeholder='Enter patient name (optional)',
            description='Patient:',
            layout=widgets.Layout(width='400px')
        )

        self.upload_btn.on_click(self._handle_upload)
        self.analyze_btn.on_click(self._handle_analysis)
        self.download_btn.on_click(self._handle_download)

    def _handle_upload(self, btn):
        """Handle file upload"""
        with self.output_area:
            clear_output()
            print("📤 Please select a CT scan image...")
            uploaded = files.upload()

            if uploaded:
                filename = list(uploaded.keys())[0]
                self.uploaded_file = filename
                self.analyze_btn.disabled = False

                img = Image.open(filename)
                plt.figure(figsize=(6, 6))
                plt.imshow(img, cmap='gray')
                plt.title(f"Uploaded: {filename}")
                plt.axis('off')
                plt.show()

                print(f"\n✅ Image uploaded successfully!")
                print("Click 'Analyze Image' to start the diagnosis.")

    def _handle_analysis(self, btn):
        """Perform AI analysis"""
        with self.output_area:
            clear_output(wait=True)

            display(HTML("""
                <div style="text-align: center; padding: 40px;">
                    <h3 style="color: #1976d2; margin-top: 20px;">Analyzing CT Scan...</h3>
                    <p style="color: #757575;">AI model is processing the image</p>
                </div>
            """))

            import time
            time.sleep(1)
            clear_output(wait=True)

            # Load and preprocess — normalized array for model
            img = image.load_img(self.uploaded_file, target_size=(224, 224))
            img_array = image.img_to_array(img)
            img_array_normalized = np.expand_dims(img_array, axis=0) / 255.0

            # Prediction
            prediction = self.model.predict(img_array_normalized, verbose=0)
            predicted_index = np.argmax(prediction[0])
            predicted_class = CLASS_NAMES[predicted_index]
            confidence = float(prediction[0][predicted_index] * 100)

            self.prediction_result = {
                'class': predicted_class,
                'confidence': confidence,
                'all_probabilities': prediction[0]
            }

            # Grad-CAM — using backbone.input pattern from working main notebook
            try:
                heatmap = make_gradcam_heatmap(img_array_normalized, self.model)
                superimposed = overlay_heatmap(img_array / 255.0, heatmap)
            except Exception as e:
                print(f"⚠️ Grad-CAM failed: {e}")
                superimposed = np.uint8(img_array)

            # Save for PDF — superimposed is uint8 RGB, save directly
            self.original_img_path = "temp_original.png"
            self.gradcam_img_path = "temp_gradcam.png"
            plt.imsave(self.original_img_path, np.uint8(img_array))
            plt.imsave(self.gradcam_img_path, superimposed)

            self._display_results(img_array / 255.0, superimposed, predicted_class, confidence)
            self.download_btn.disabled = False

    def _display_results(self, original_img, gradcam_img, predicted_class, confidence):
        """Display prediction results"""
        condition_info = KIDNEY_KNOWLEDGE_BASE.get(predicted_class, {})
        urgency_color = condition_info.get('urgency_color', '#757575')

        display(HTML(f"""
            <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        padding: 30px; border-radius: 15px; color: white; text-align: center;
                        margin-bottom: 30px; box-shadow: 0 8px 16px rgba(0,0,0,0.2);">
                <h1 style="margin: 0; font-size: 32px;">🩺 AI Diagnostic Results</h1>
                <p style="margin: 10px 0 0 0; font-size: 16px; opacity: 0.9;">
                    Analysis completed on {datetime.now().strftime("%B %d, %Y at %I:%M %p")}
                </p>
            </div>
        """))

        display(HTML(f"""
            <div style="background: white; border: 3px solid {urgency_color};
                        border-radius: 15px; padding: 25px; margin-bottom: 30px;
                        box-shadow: 0 4px 12px rgba(0,0,0,0.15);">
                <div style="text-align: center;">
                    <h2 style="color: {urgency_color}; font-size: 36px; margin: 0;">{predicted_class}</h2>
                    <p style="color: #757575; font-size: 18px; margin: 10px 0;">
                        Confidence: <strong>{confidence:.1f}%</strong>
                    </p>
                    <p style="color: #424242; font-size: 16px; margin: 15px 0; padding: 15px;
                               background: #f5f5f5; border-radius: 8px;">
                        <strong>Urgency Level:</strong> {condition_info.get('urgency_level', 'N/A')}
                    </p>
                </div>
                <hr style="border: none; border-top: 2px solid #e0e0e0; margin: 20px 0;">
                <p style="color: #424242; font-size: 14px; line-height: 1.6; text-align: justify;">
                    <strong>Summary:</strong> {condition_info.get('summary', '')}
                </p>
            </div>
        """))

        # gradcam_img is uint8 RGB — display directly, no division
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(original_img)
        axes[0].set_title('Original CT Scan', fontsize=14, fontweight='bold')
        axes[0].axis('off')
        axes[1].imshow(gradcam_img)
        axes[1].set_title('Grad-CAM Visualization', fontsize=14, fontweight='bold')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

        self._show_probability_chart()
        self._show_quick_reference(predicted_class)

    def _show_probability_chart(self):
        """Display probability distribution chart"""
        probabilities = self.prediction_result['all_probabilities']

        fig, ax = plt.subplots(figsize=(10, 5))
        colors_bar = ['#4caf50' if CLASS_NAMES[i] == self.prediction_result['class'] else '#e0e0e0'
                      for i in range(len(CLASS_NAMES))]

        bars = ax.barh(CLASS_NAMES, probabilities * 100, color=colors_bar)
        ax.set_xlabel('Confidence (%)', fontsize=12, fontweight='bold')
        ax.set_title('Prediction Confidence for All Classes', fontsize=14, fontweight='bold')
        ax.set_xlim(0, 100)

        for i, (bar, prob) in enumerate(zip(bars, probabilities)):
            width = bar.get_width()
            ax.text(width + 2, bar.get_y() + bar.get_height()/2,
                   f'{prob*100:.1f}%',
                   ha='left', va='center', fontweight='bold')

        plt.tight_layout()
        plt.show()

    def _show_quick_reference(self, predicted_class):
        """Show clinical quick reference"""
        condition = KIDNEY_KNOWLEDGE_BASE.get(predicted_class, {})
        urgency_color = condition.get('urgency_color', '#757575')

        next_steps_html = "".join([
            f"<li style='margin-bottom: 8px;'>{step}</li>"
            for step in condition.get('next_steps', [])[:5]
        ])

        display(HTML(f"""
            <div style="background: white; border-left: 5px solid {urgency_color};
                        padding: 20px; margin: 20px 0; border-radius: 8px;
                        box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
                <h3 style="color: {urgency_color}; margin-top: 0;">📋 Recommended Next Steps</h3>
                <ol style="line-height: 1.8; color: #424242;">
                    {next_steps_html}
                </ol>
                <div style="background: #fff3e0; padding: 15px; border-radius: 8px; margin-top: 15px;">
                    <p style="margin: 0; color: #e65100;">
                        <strong>⏱️ Follow-up Timeline:</strong> {condition.get('follow_up', '')}
                    </p>
                </div>
            </div>
        """))

    def _handle_download(self, btn):
        """Generate and download PDF report"""
        with self.output_area:
            print("\n📄 Generating PDF report...")

            patient_name = self.patient_name_input.value.strip() or "Patient"

            report_gen = KidneyDiagnosticReport(patient_name=patient_name)
            pdf_filename = report_gen.generate_report(
                predicted_class=self.prediction_result['class'],
                confidence=self.prediction_result['confidence'],
                original_img_path=self.original_img_path,
                gradcam_img_path=self.gradcam_img_path,
                filename="kidney_diagnostic_report.pdf"
            )

            files.download(pdf_filename)
            print("\n✅ PDF report downloaded successfully!")

            print("\n" + "="*80)
            print("💬 INTERACTIVE PATIENT EDUCATION")
            print("="*80)

            chatbot = ImprovedKidneyChatbot(
                self.prediction_result['class'],
                self.prediction_result['confidence']
            )
            chatbot.display()

    def display(self):
        """Display the complete UI"""
        header = widgets.HTML("""
            <div style="background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);
                        padding: 40px; border-radius: 15px; color: white; text-align: center;
                        margin-bottom: 30px; box-shadow: 0 10px 25px rgba(0,0,0,0.3);">
                <h1 style="margin: 0; font-size: 42px; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">
                    🏥 AI-Powered Kidney Lesion Classification System
                </h1>
                <p style="margin: 15px 0 0 0; font-size: 18px; opacity: 0.95;">
                    Automated CT Scan Analysis with Explainable AI
                </p>
            </div>
        """)

        controls = widgets.VBox([
            widgets.HTML("<h3 style='color: #424242; margin-bottom: 15px;'>📝 Patient Information</h3>"),
            self.patient_name_input,
            widgets.HTML("<br><h3 style='color: #424242; margin: 20px 0 15px 0;'>🔬 Analysis Controls</h3>"),
            widgets.HBox([self.upload_btn, self.analyze_btn, self.download_btn],
                        layout=widgets.Layout(justify_content='space-around')),
        ], layout=widgets.Layout(
            border='2px solid #e0e0e0',
            border_radius='10px',
            padding='25px',
            margin='20px 0',
            background='#fafafa'
        ))

        full_ui = widgets.VBox([
            header,
            controls,
            self.output_area
        ])

        display(full_ui)

print("✅ Complete UI system loaded!")


✅ Complete UI system loaded!


---
## 🚀 LAUNCH THE SYSTEM

In [ ]:
# Create and display the complete system
diagnosis_ui = KidneyDiagnosisUI(model)
diagnosis_ui.display()

print("\n" + "="*80)
print("🎉 SYSTEM READY!")
print("="*80)
print("\n📋 Instructions:")

print("1. Enter patient name (optional)")
print("2. Click 'Upload CT Scan' to select an image")
print("3. Click 'Analyze Image' to run AI diagnosis")
print("4. Review results and click 'Download PDF Report'")
print("5. Use the chatbot to ask questions about the diagnosis")
print("\n✨ Enjoy your complete diagnostic system!\n")


🎉 SYSTEM READY!

📋 Instructions:
1. Enter patient name (optional)
2. Click 'Upload CT Scan' to select an image
3. Click 'Analyze Image' to run AI diagnosis
4. Review results and click 'Download PDF Report'
5. Use the chatbot to ask questions about the diagnosis

✨ Enjoy your complete diagnostic system!

